In [ ]:
# Feature filtering 
def filter_features(X_train, 
                    nonzero_thresh=0.01,    # non-zero ratio threshold
                    var_thresh=1e-6,         # variance threshold
                    corr_thresh=0.98,        # correlation coefficient threshold
                    sample_ratio=0.1,         # sampling ratio for correlation computation
                    eps=1e-12):               # tolerance for zero value detection

    n_features = X_train.shape[1]
    mask = np.ones(n_features, dtype=bool)

    # Non-zero ratio filtering
    nonzero_ratio = np.mean(np.abs(X_train) > eps, axis=0)
    mask &= (nonzero_ratio >= nonzero_thresh)
    print(f"Non-zero ratio filtering: features remaining = {mask.sum()}")

    # Variance filtering
    variances = np.var(X_train, axis=0)
    mask &= (variances >= var_thresh)
    print(f"Variance filtering: features remaining = {mask.sum()}")

    # High correlation filtering (on remaining features, with sampling for speed)
    # Get current retained feature indices and corresponding training data
    remain_idx = np.where(mask)[0]
    if len(remain_idx) == 0:
        print("Warning: all features filtered, check thresholds!")
        return mask

    # Sampling
    n_samples = X_train.shape[0]
    sample_size = min(int(n_samples * sample_ratio), 10000)
    if sample_size < n_samples:
        np.random.seed(42)  # ensure reproducibility
        sample_idx = np.random.choice(n_samples, size=sample_size, replace=False)
        X_sample = X_train[sample_idx][:, remain_idx]
    else:
        X_sample = X_train[:, remain_idx]

    # Compute correlation matrix
    corr_matrix = np.corrcoef(X_sample, rowvar=False)
    corr_matrix = np.nan_to_num(corr_matrix)  # handle possible NaNs

    # Keep the first occurrence, remove subsequent ones that are highly correlated
    n_remain = len(remain_idx)
    keep = np.ones(n_remain, dtype=bool)
    for i in range(n_remain):
        if not keep[i]:
            continue
        for j in range(i + 1, n_remain):
            if not keep[j]:
                continue
            if abs(corr_matrix[i, j]) > corr_thresh:
                keep[j] = False

    # Map filtering results back to the original feature space
    final_mask = mask.copy()
    final_mask[remain_idx[~keep]] = False
    print(f"High correlation filtering: features remaining = {final_mask.sum()}")

    return final_mask

In [ ]:
# Perform feature filtering based on training set
X_train_raw = merged_raw[train_indices]          # raw features of training set
feature_mask = filter_features(X_train_raw)

# Apply feature mask to all data
merged_raw = merged_raw[:, feature_mask]

# Save feature mask for prediction
joblib.dump(feature_mask, 'feature_mask.pkl')

# Fit StandardScaler on training set (merged_raw already filtered)
scaler = StandardScaler()
scaler.fit(merged_raw[train_indices])
merged_scaled = scaler.transform(merged_raw)  # (N, new number of features)

# Save scaler for later prediction
joblib.dump(scaler, 'scaler_merged.pkl')

In [ ]:
# Prepare training and test data
train_matrices, train_labels, train_indexes = prepare_data(train_indices)
test_matrices, test_labels, test_indexes = prepare_data(test_indices)

# Dataset definition
class RockDataset(Dataset):
    def __init__(self, matrices, labels, indexes, training=True):
        self.matrices = matrices
        self.labels = labels
        self.indexes = indexes
        self.training = training

    def __len__(self):
        return len(self.matrices)

    def __getitem__(self, idx):
        matrix = self.matrices[idx]
        label = self.labels[idx]
        index = self.indexes[idx]
        if self.training:
            matrix = self.random_drop(matrix)
        return matrix, label, index

    def random_drop(self, matrix, drop_prob=0.2, drop_ratio=0.2):
        if random.random() > drop_prob:
            return matrix
        C, D, H, W = matrix.shape
        drop_d = int(D * drop_ratio)
        drop_h = int(H * drop_ratio)
        drop_w = int(W * drop_ratio)
        x = random.randint(0, D - drop_d)
        y = random.randint(0, H - drop_h)
        z = random.randint(0, W - drop_w)
        mask = torch.ones_like(matrix)
        mask[:, x:x+drop_d, y:y+drop_h, z:z+drop_w] = 0
        return matrix * mask

train_dataset = RockDataset(train_matrices, train_labels, train_indexes, training=True)
test_dataset = RockDataset(test_matrices, test_labels, test_indexes, training=False)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
# Compute class weights
train_labels_np = train_labels.cpu().numpy()
classes = np.unique(train_labels_np)
class_counts = np.array([np.sum(train_labels_np == c) for c in classes])

inv_counts = 1.0 / class_counts
normalized_weights = inv_counts / np.sum(inv_counts)

# Convert to PyTorch tensor
class_weights = torch.tensor(normalized_weights, dtype=torch.float32).to(device)

In [ ]:
# Early stopping mechanism
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0):
        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False
    def __call__(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

# Loss function
class LabelSmoothingFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.reduction = reduction
    def forward(self, inputs, targets):
        num_classes = inputs.size(-1)
        log_preds = F.log_softmax(inputs, dim=-1)
        with torch.no_grad():
            smooth_targets = torch.zeros_like(log_preds)
            smooth_targets.fill_(self.smoothing / (num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        kl_loss = F.kl_div(log_preds, smooth_targets, reduction='none').sum(-1)
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        p = torch.exp(-ce_loss)
        focal_weight = (1 - p) ** self.gamma
        loss = focal_weight * kl_loss
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AnisotropicConv3d(nn.Module):
    """
    Anisotropic convolution module: performs depthwise convolution along X, Y, Z directions separately,
    then mixes channels via a 1x1x1 convolution.
    :param in_channels: number of input channels
    :param out_channels: number of output channels
    :param kernel_size: triplet (kx, ky, kz), kernel size in each direction
    :param stride: stride (default 1; if >1, same stride in all directions for simplicity)
    :param padding: triplet (padx, pady, padz), padding in each direction
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(AnisotropicConv3d, self).__init__()
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size, kernel_size)
        if isinstance(padding, int):
            padding = (padding, padding, padding)

        # Depthwise convolution: along X direction (kx,1,1)
        self.conv_x = nn.Conv3d(
            in_channels, in_channels,
            kernel_size=(kernel_size[0], 1, 1),
            stride=stride,
            padding=(padding[0], 0, 0),
            groups=in_channels,
            bias=False
        )
        # Depthwise convolution: along Y direction (1,ky,1)
        self.conv_y = nn.Conv3d(
            in_channels, in_channels,
            kernel_size=(1, kernel_size[1], 1),
            stride=stride,
            padding=(0, padding[1], 0),
            groups=in_channels,
            bias=False
        )
        # Depthwise convolution: along Z direction (1,1,kz)
        self.conv_z = nn.Conv3d(
            in_channels, in_channels,
            kernel_size=(1, 1, kernel_size[2]),
            stride=stride,
            padding=(0, 0, padding[2]),
            groups=in_channels,
            bias=False
        )
        # Pointwise convolution (channel mixing)
        self.pointwise = nn.Conv3d(in_channels, out_channels, kernel_size=1, bias=True)

    def forward(self, x):
        x = self.conv_x(x)
        x = self.conv_y(x)
        x = self.conv_z(x)
        x = self.pointwise(x)
        return x

In [ ]:
class LeNet5_3D(nn.Module):
    def __init__(self, in_channels, num_classes, dropout_rate=0.5):
        super(LeNet5_3D, self).__init__()
        # Anisotropic convolutional layers
        self.conv1 = AnisotropicConv3d(in_channels=in_channels, out_channels=32,kernel_size=(3, 5, 5),stride=1,padding=(1, 2, 2))
        self.bn1 = nn.BatchNorm3d(32)

        self.conv2 = AnisotropicConv3d(in_channels=32,out_channels=32,kernel_size=(3, 5, 5),stride=1,padding=(1, 2, 2))
        self.bn2 = nn.BatchNorm3d(32)

        self.conv3 = AnisotropicConv3d(in_channels=32,out_channels=64,kernel_size=(3, 5, 5),stride=1,padding=(1, 2, 2))
        self.bn3 = nn.BatchNorm3d(64)

        self.conv4 = AnisotropicConv3d(in_channels=64,out_channels=128,kernel_size=(3, 5, 5),stride=1,padding=(1, 2, 2))
        self.bn4 = nn.BatchNorm3d(128)

        self.conv5 = AnisotropicConv3d(in_channels=128,out_channels=256,kernel_size=(3, 5, 5),stride=1,padding=(1, 2, 2))
        self.bn5 = nn.BatchNorm3d(256)
        
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2, padding=1)
        self.global_avg_pool = nn.AdaptiveAvgPool3d(1)
        
        self.fc1 = nn.Linear(256, 512)
        self.bn8 = nn.BatchNorm1d(512)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = F.leaky_relu(self.bn1(self.conv1(x)), 0.01)
        x = F.leaky_relu(self.bn2(self.conv2(x)), 0.01)
        x = self.pool(x)
        
        x = F.leaky_relu(self.bn3(self.conv3(x)), 0.01)
        x = F.leaky_relu(self.bn4(self.conv4(x)), 0.01)
        x = self.pool(x)
        
        x = F.leaky_relu(self.bn5(self.conv5(x)), 0.01)
        x = self.pool(x)
        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)
        x = F.leaky_relu(self.bn8(self.fc1(x)), 0.01)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
# Initialize model
num_classes = 3
in_channels = merged_scaled.shape[1]   # automatically adapt to filtered feature dimension
model = LeNet5_3D(in_channels, num_classes).to(device)
model = nn.DataParallel(model, device_ids=[0,1,2,3,4,5,6,7])
# Loss function
criterion = LabelSmoothingFocalLoss(alpha=class_weights, gamma=2.0, smoothing=0.2, reduction='mean')
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

early_stopping = EarlyStopping(patience=30, min_delta=0.001)

In [ ]:
# Training loop
train_losses, test_losses = [], []
train_accuracies, test_accuracies = [], []
num_epochs = 200
best_test_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels, indexes in train_loader:
        inputs, labels = inputs.to(device), labels.to(device).long()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}, Train Accuracy: {epoch_accuracy:.2f}%")

    # Testing
    model.eval()
    test_loss = 0.0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels, indexes in test_loader:
            inputs, labels = inputs.to(device), labels.to(device).long()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    test_loss /= len(test_loader)
    test_accuracy = balanced_accuracy_score(all_labels, all_preds) * 100
    test_losses.append(test_loss)
    test_accuracies.append(test_accuracy)
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")
    scheduler.step(test_loss)

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        state_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state_dict, '/root/autodl-tmp/best.pth')

    early_stopping(test_loss)
    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch+1}!")
        break

In [ ]:
# Create combined canvas
plt.figure(figsize=(10, 5))

# Plot loss curves (left)
plt.subplot(1, 2, 1)
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Train Loss', color='blue')
plt.plot(range(1, len(test_losses) + 1), test_losses, label='Test Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curves')
plt.legend()
plt.grid(True)

# Plot accuracy curves (right)
plt.subplot(1, 2, 2)
plt.plot(range(1, len(train_accuracies) + 1), train_accuracies, label='Train Accuracy', color='green')
plt.plot(range(1, len(test_accuracies) + 1), test_accuracies, label='Test Accuracy', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Curves')
plt.legend()
plt.grid(True)

In [ ]:
# Adjust layout and save
plt.tight_layout()
save_path = "autodl-tmp/combined_curves.png"
plt.savefig(save_path, dpi=100, bbox_inches='tight')
#print(f"Combined curves saved to: {save_path}")
#plt.show()

# Evaluation
target_accuracy = 100 * correct / total
print(f"Target Data Accuracy: {target_accuracy:.2f}%")

precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
conf_matrix = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(conf_matrix)

In [ ]:
# Prediction section
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import joblib
from scipy.spatial import cKDTree
import torch.nn.functional as F

# Load objects saved during training
knots_list = joblib.load('knots_list.pkl')          # multi-resolution knot list
feature_mask = joblib.load('feature_mask.pkl')      # feature mask
scaler = joblib.load('scaler_merged.pkl')            # standardizer

# Read new data
data_pred = pd.read_csv(r"s50_modify_data.dat")
coords_pred = data_pred[["X", "Y", "Z"]].values
features_pred_raw = data_pred[["den", "sus", "res"]].values

# Compute multi-resolution basis functions (consistent with training)
basis_pred_list = []
for knots in knots_list:
    basis = compute_basis_values(coords_pred, knots, theta_factor=1.2)
    basis_pred_list.append(basis)
basis_pred_multi = np.concatenate(basis_pred_list, axis=1)

# Concatenate raw features and basis functions
merged_pred_raw = np.concatenate([features_pred_raw, basis_pred_multi], axis=1)

# Apply feature mask (keep only features selected during training)
merged_pred_raw = merged_pred_raw[:, feature_mask]

# Standardize
merged_pred_scaled = scaler.transform(merged_pred_raw)

# Build KDTree and neighbor indices (consistent with training)
N = 2196  # number of neighbors, consistent with training
kd_tree_pred = cKDTree(coords_pred)
neighbors_indices_pred = [kd_tree_pred.query(point, k=N + 1)[1][1:] for point in coords_pred]

# Define a dataset that returns indices only
class IndexDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        return self.indices[idx]

batch_size = 256
pred_indices = np.arange(len(data_pred))
index_dataset = IndexDataset(pred_indices)
index_loader = DataLoader(index_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# Load the best model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_channels = merged_pred_scaled.shape[1] 
num_classes = 3  # adjust according to the actual task
model = LeNet5_3D(in_channels, num_classes).to(device)
model.load_state_dict(torch.load('/root/autodl-tmp/best.pth'))
model.eval()

prediction_data = []
file_path = 'prediction_RBF_ACNN.csv'
total_samples = len(pred_indices)
processed = 0

with torch.no_grad():
    for batch_indices_tensor in index_loader:
        batch_indices = batch_indices_tensor.numpy()

        # Build neighbor index matrix for samples in this batch (batch_size, 2197)
        full_indices = np.array(
            [[i] + neighbors_indices_pred[i].tolist() for i in batch_indices],
            dtype=np.int32
        )

        # Extract corresponding rows from the standardized feature matrix → (batch_size, 2197, 92)
        matrices = merged_pred_scaled[full_indices]

        # Reshape to (batch_size, 13, 13, 13, 92) and transpose to (batch_size, 92, 13, 13, 13)
        matrices = matrices.reshape(-1, 13, 13, 13, merged_pred_scaled.shape[1])
        matrices = np.transpose(matrices, (0, 4, 1, 2, 3))

        # Convert to tensor and move to GPU
        inputs = torch.tensor(matrices, dtype=torch.float32).to(device)

        # Inference
        outputs = model(inputs)
        probs = F.softmax(outputs, dim=1).cpu().numpy()
        _, predicted = torch.max(outputs, 1)

        # Get original data rows
        original_rows = data_pred.iloc[batch_indices].reset_index(drop=True)
        batch_df = original_rows.copy()
        # Remove possible existing columns
        for col in batch_df.columns:
            if col.startswith('class_') or col == "YXML50":
                batch_df.drop(columns=[col], inplace=True, errors='ignore')

        # Add prediction results
        batch_df['predicted_class'] = predicted.cpu().numpy()
        prob_df = pd.DataFrame(probs, columns=[f'class_{i}' for i in range(num_classes)])
        batch_df = pd.concat([batch_df, prob_df], axis=1)

        prediction_data.append(batch_df)
        processed += len(batch_indices)
        print(f"Processed {processed}/{total_samples} samples", end='\r')

# Concatenate all batch results and save
prediction_df = pd.concat(prediction_data, ignore_index=True)
prediction_df.to_csv(file_path, index=False, encoding='utf-8-sig')